In [1]:
import time

notebook_start_time = time.time()

# Set up environment

In [2]:
import sys
from pathlib import Path


def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False


def clone_repository() -> None:
    !git clone https://github.com/decodingml/hands-on-recommender-system.git
    %cd hands-on-recommender-system/


def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()

    root_dir = str(Path().absolute())
    print("⛳️ Google Colab environment")
else:
    root_dir = str(Path().absolute().parent)
    print("⛳️ Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    print(f"Adding the following directory to the PYTHONPATH: {root_dir}")
    sys.path.append(root_dir)

⛳️ Local environment
Adding the following directory to the PYTHONPATH: c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course


# Offline inference pipeline: Computing item embeddings

In this notebook we will compute the candidate embeddings and populate a Hopsworks feature group with a vector index.

## Imports

In [3]:
import warnings

warnings.filterwarnings("ignore")

from loguru import logger

from recsys import features, hopsworks_integration
from recsys.config import settings

## Constants

In [4]:
from pprint import pprint

pprint(dict(settings))

{'CUSTOMER_DATA_SIZE': <CustomerDatasetSize.SMALL: 'SMALL'>,
 'CUSTOM_HOPSWORKS_INFERENCE_ENV': 'custom_env_name',
 'FEATURES_EMBEDDING_MODEL_ID': 'all-MiniLM-L6-v2',
 'HOPSWORKS_API_KEY': SecretStr('**********'),
 'OPENAI_API_KEY': SecretStr(''),
 'OPENAI_MODEL_ID': 'gpt-4o-mini',
 'RANKING_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'RANKING_EARLY_STOPPING_ROUNDS': 5,
 'RANKING_ITERATIONS': 100,
 'RANKING_LEARNING_RATE': 0.2,
 'RANKING_MODEL_TYPE': 'ranking',
 'RANKING_SCALE_POS_WEIGHT': 10,
 'RECSYS_DIR': WindowsPath('c:/Users/PC/OneDrive - National Economics University/Tài liệu/personalized-recommender-course/recsys'),
 'TWO_TOWER_DATASET_TEST_SPLIT_SIZE': 0.1,
 'TWO_TOWER_DATASET_VALIDATON_SPLIT_SIZE': 0.1,
 'TWO_TOWER_LEARNING_RATE': 0.01,
 'TWO_TOWER_MODEL_BATCH_SIZE': 2048,
 'TWO_TOWER_MODEL_EMBEDDING_SIZE': 16,
 'TWO_TOWER_NUM_EPOCHS': 10,
 'TWO_TOWER_WEIGHT_DECAY': 0.001}


## Connect to Hopsworks Feature Store 

In [5]:
import hopsworks

hopsworks_host = "eu-west.cloud.hopsworks.ai"

project = hopsworks.login(
    host=hopsworks_host,
    api_key_value=settings.HOPSWORKS_API_KEY.get_secret_value()
)

# Thêm logic này để tự tạo project nếu tài khoản bị trống
if project is None:
    project = hopsworks.create_project("recommender_system", description="H&M Recommender")

fs = project.get_feature_store()

mr = project.get_model_registry()

2026-03-10 20:36:26,464 INFO: Initializing external client
2026-03-10 20:36:26,467 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-10 20:36:30,214 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873


# Computing candidate embeddings

You start by computing candidate embeddings for all items in the training data.

First, you load your candidate model. Recall that you uploaded it to the Hopsworks Model Registry in previous steps:

In [6]:
candidate_model, candidate_features = (
    hopsworks_integration.two_tower_serving.HopsworksCandidateModel.download(mr=mr)
)

2026-03-10 20:36:32.755 | INFO     | recsys.hopsworks_integration.two_tower_serving:download:198 - Downloading 'candidate_model' version 1
Downloading: 100.000%|██████████| 423/423 elapsed<00:00 remaining<00:00


Downloading: 100.000%|██████████| 766954/766954 elapsed<00:03 remaining<00:00


Downloading: 100.000%|██████████| 326252/326252 elapsed<00:01 remaining<00:00


Downloading: 100.000%|██████████| 55/55 elapsed<00:00 remaining<00:00


### Get candidates data

Now, we get the training retrieval data containing all the features required for the candidate embedding model.

In [7]:
feature_view = fs.get_feature_view(
    name="retrieval",
    version=1,
)

In [8]:
train_df, val_df, test_df, _, _, _ = feature_view.train_validation_test_split(
    validation_size=settings.TWO_TOWER_DATASET_VALIDATON_SPLIT_SIZE,
    test_size=settings.TWO_TOWER_DATASET_TEST_SPLIT_SIZE,
    description="Retrieval dataset splits",
)
train_df.head(3)

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (27.19s) 
2026-03-10 20:37:27,277 INFO: Computing insert statistics
2026-03-10 20:37:27,418 INFO: Computing insert statistics
2026-03-10 20:37:27,495 INFO: Computing insert statistics


,customer_id,article_id,t_dat,price,month_sin,month_cos,age,club_member_status,age_group,garment_group_name,index_group_name
0,6d4c6e47fe5933f9d317263d19b5685223c4bedd160c1a...,711649001,0,0.025407,-2.449294e-16,1.000000e+00,33.0,ACTIVE,26-35,"Under-, Nightwear",Ladieswear
1,8e6338ee4363c6378f4a07b15356c440a446ab2aff6aa2...,846347005,0,0.025407,-5.000000e-01,-8.660254e-01,52.0,ACTIVE,46-55,Skirts,Ladieswear
2,b4397f1ef360771e411afa9b43ad9ad0e46d4b9a6d6cf2...,678963010,0,0.050831,1.000000e+00,6.123234e-17,29.0,ACTIVE,26-35,Outdoor,Divided


### Compute embeddings

Next you compute the embeddings of all candidate items that were used to train the retrieval model.

In [9]:
item_df = features.embeddings.preprocess(train_df, candidate_features)
item_df.head(3)

,garment_group_name,article_id,index_group_name
0,"Under-, Nightwear",711649001,Ladieswear
1,Skirts,846347005,Ladieswear
2,Outdoor,678963010,Divided


In [10]:
embeddings_df = features.embeddings.embed(df=item_df, candidate_model=candidate_model)
embeddings_df.head()

,article_id,embeddings
0,711649001,"[-1.148377776145935, -0.7675642371177673, -2.1..."
1,846347005,"[2.2024989128112793, 0.4310351312160492, -1.14..."
2,678963010,"[-0.4776822328567505, -0.6946502923965454, -1...."
3,734316001,"[2.127501964569092, 1.2981139421463013, -0.735..."
4,429313013,"[1.6846195459365845, 1.5434237718582153, -0.88..."


# Create Hopsworks Embedding Index </span>

Now we are ready to create a feature group for your candidate embeddings.

In [11]:
candidate_embeddings_fg = (
    hopsworks_integration.feature_store.create_candidate_embeddings_feature_group(
        fs=fs, df=embeddings_df, online_enabled=True
    )
)
logger.info("✅ Uploaded 'candidate_embeddings' Feature Group to Hopsworks!!")

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31873/fs/20557/fg/37018


Uploading Dataframe: 100.00% |██████████| Rows 11839/11839 | Elapsed Time: 00:03 | Remaining Time: 00:00


Launching job: candidate_embeddings_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31873/jobs/named/candidate_embeddings_1_offline_fg_materialization/executions
2026-03-10 20:38:01,217 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-03-10 20:38:04,497 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-03-10 20:40:17,405 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-03-10 20:40:20,697 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-03-10 20:40:21,653 INFO: Waiting for log aggregation to finish.
2026-03-10 20:40:31,046 INFO: Execution finished successfully.


Online data ingestion progress: 100.00% |██████████| Rows 11839/11839
2026-03-10 20:40:31.375 | INFO     | __main__:<module>:6 - ✅ Uploaded 'candidate_embeddings' Feature Group to Hopsworks!!


## Expose it to the online inference pipeline as a Feature View


In [12]:
feature_view = (
    hopsworks_integration.feature_store.create_candidate_embeddings_feature_view(
        fs=fs, fg=candidate_embeddings_fg
    )
)

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31873/fs/20557/fv/candidate_embeddings/version/1


## Inspecting the embeddings in Hopsworks UI </span>

View results in [Hopsworks Serverless](https://rebrand.ly/serverless-github): **Feature Store → Feature Groups**

---

In [13]:
notebook_end_time = time.time()
notebook_execution_time = notebook_end_time - notebook_start_time

logger.info(
    f"⌛️ Notebook Execution time: {notebook_execution_time:.2f} seconds ~ {notebook_execution_time / 60:.2f} minutes"
)

2026-03-10 20:40:35.793 | INFO     | __main__:<module>:4 - ⌛️ Notebook Execution time: 287.21 seconds ~ 4.79 minutes
